# Agents

Các tác nhân (agent) kết hợp mô hình ngôn ngữ với các công cụ để tạo ra những hệ thống có thể suy luận về nhiệm vụ, quyết định nên sử dụng công cụ nào và làm việc theo từng bước để tiến tới lời giải.

Một LLM Agent vận hành các công cụ theo vòng lặp nhằm đạt được mục tiêu. Tác nhân sẽ tiếp tục chạy cho đến khi đạt điều kiện dừng — ví dụ như khi mô hình tạo ra kết quả cuối cùng hoặc khi chạm đến giới hạn số vòng lặp.

![Agent](./images/agent.png)

# Tools 
Các công cụ (tools) trao cho agent khả năng thực hiện hành động. Agent vượt xa việc chỉ đơn thuần “gắn” mô hình với công cụ bằng cách hỗ trợ:
- Gọi nhiều công cụ theo chuỗi (được kích hoạt từ một prompt duy nhất)
- Gọi công cụ song song khi phù hợp
- Lựa chọn công cụ một cách động dựa trên các kết quả trước đó
- Cơ chế thử lại công cụ và xử lý lỗi

Duy trì trạng thái xuyên suốt các lần gọi công cụ

## Defining tools
Pass a list of tools to the agent.

In [1]:
from langchain.tools import tool
from langchain.agents import create_agent

### Search Tool 
https://docs.langchain.com/oss/python/integrations/tools


In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Rikkeisoft?")

"December 21, 2025 - Rikkeisoft (Công ty Cổ phần Rikkeisoft) là công ty công nghệ thông tin tại Việt Nam được thành lập vào năm 2012 với lĩnh vực kinh doanh chính là cung cấp các dịch vụ và giải pháp công nghệ thông tin . Hiện tại, Rikkeisoft có 05 văn phòng làm ... October 14, 2025 - Rikkeisoft（本社：ベトナム・ハノイ、代表取締役会長：Ta Son Tung）は、2025年10月10日にハノイで開催された国際テクノロジーサミット「Rikkei Global Summit 2025（以下、RGS2025）」で、新たなグローバル戦略を発表しました。 October 16, 2025 - We appreciate your interest in Rikkeisoft. Send us a question, and we'll get back to you as soon as possible. May 29, 2025 - After a successful trial phase demonstrating substantial improvements in productivity and product quality, Rikkeisoft will officially integrate AI into all business operations starting June 1, 2025. October 10, 2025 - The true value of AI lies not in replacing humans, but in liberating them from repetitive work — so they can focus on creativity and strategic decisions. Guided by this principle, Rikkeisoft's integrated AI suite e

In [6]:
from langchain_community.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults(output_format="list")

search.invoke("Rikkeisoft?")

[{'snippet': 'We have been working with Rikkeisoft for 7 years, mainly in web and mobile application development. Rikkeisoft team can communicate fluently in Japanese and English, so we are assured of mutual understanding. In the future, we still want to keep a long-term relationship with Rikkeisoft .',
  'title': 'RIKKEI (THAILAND) CO., LTD - Rikkeisoft - Trusted IT Outsourcing Provider',
  'link': 'https://rikkeisoft.com/th/th/'},
 {'snippet': '231+ reviews môi trường làm việc, văn hoá, mức lương tại RIKKEISOFT . Được đăng ẩn danh bởi nhân viên làm việc tại đây',
  'title': '231+ Reviews RIKKEISOFT: Công ty có tốt không?',
  'link': 'https://1900.com.vn/danh-gia-dn/cong-ty-co-phan-rikkeisoft-1657'},
 {'snippet': 'Ông Tạ Sơn Tùng, Chủ tịch Rikkeisoft chia sẻ, Rikkeisoft đang chuẩn bị cho cột mốc IPO tại Nhật trong 3 năm tới và hướng tới giấc mơ trở thành kỳ lân công nghệ Việt Nam.',
  'title': 'Rikkeisoft tuyên bố IPO tại Nhật, hướng tới trở thành kỳ lân công nghệ ...',
  'link': 'htt

In [5]:
@tool
def search_rikkeisoft_information(query: str) -> str:
    """Search for information about Rikkeisoft."""
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    result = search.invoke(query)
    if result:
        return result
    else:
        return "No information found"

search_rikkeisoft_information.invoke({
    "query": "Rikkeisoft?"
})





'Rikkeisoft (Công ty Cổ phần Rikkeisoft ) là công ty công nghệ thông tin tại Việt Nam được thành lập vào năm 2012 với lĩnh vực kinh doanh chính là cung cấp các dịch vụ và giải pháp công nghệ thông tin. In 2012, Rikkeisoft was established by six software developers on a lofty mission: to drive technological innovation and deliver lasting values. Following in the footsteps of our founders, we cherish the tradition of excellence. Leading Software solutions and Services provider in Vietnam. Founded in 2012, Rikkeisoft is an award-winning, leading Vietnamese IT Enterprise. We help our partner companies grow sustainably with... Rikkeisoft , tập trung vào đào tạo và phát triển nguồn nhân lực, cung cấp các chương trình đào tạo gắn với thực tiễn công nghệ và nhu cầu thị trường, hướng tới phát triển nguồn nhân lực chất lượng cao cho doanh nghiệp trong nước và quốc tế. Established in 2012, Rikkeisoft is a leading provider of technology resources & services for the US, Europe, and Asia-Pacific (AP

### Get current time tool

In [14]:
@tool(description="Get current time by UTC offset.")
def get_current_time(utc: int = 0) -> str:
    """Get current time by UTC offset."""
    from datetime import datetime, timedelta, timezone

    tz = timezone(timedelta(hours=utc))
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")


In [7]:

@tool
def calculate_years_of_establishment(start_year: int) -> str:
    """Calculate the number of years since the establishment of Rikkeisoft."""
    from datetime import datetime
    return f"Rikkeisoft was established in {datetime.now().year - start_year} years ago"

calculate_years_of_establishment.invoke({
    "start_year": 2010
})

'Rikkeisoft was established in 16 years ago'

## Model
The model is the reasoning engine of your agent. It can be specified in multiple ways, supporting both static and dynamic model selection.

### Static model
Static models are configured once when creating the agent and remain unchanged throughout execution. This is the most common and straightforward approach.

In [15]:
from config import settings
from langchain.agents import create_agent
from  langchain_openai import ChatOpenAI
model = ChatOpenAI(model=settings.LLM_CHAT_MODEL,api_key=settings.LLM_API_KEY, base_url=settings.LLM_BASE_URL)

agent = create_agent(model=model, tools=[
    search_rikkeisoft_information,
    get_current_time,
    calculate_years_of_establishment
])


In [16]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time in Vietnam?"}]}
)

In [17]:
result

{'messages': [HumanMessage(content='what is the current time in Vietnam?', additional_kwargs={}, response_metadata={}, id='e66c17ff-d4d1-49fa-ad2f-5ccb48e73e1b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 160, 'total_tokens': 176, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 160}}, 'model_provider': 'openai', 'model_name': 'gemini-3-flash-preview', 'system_fingerprint': None, 'id': 'fFCBaZ6WKtKj1e8P0ZmBkQo', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c2122-61f1-7cd2-87f6-fed3022eab99-0', tool_calls=[{'name': 'get_current_time', 'args': {'utc': 7}, 'id': 'call_538b741d5e8648d38b5668bda707__thought__EjQKMgFyyNp8AaRzTuF6SH2FpmapEUC14dTLQ4t20+gvDpTMwaAXkAkDiNWcDRdN174MCWhi', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 160, 'output_tokens': 16, 'total_tokens': 176, '

In [18]:
final_message = result["messages"][-1].content
print(final_message)

The current time in Vietnam is 8:33 AM on Tuesday, February 3, 2026 (GMT+7).


In [20]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]}
)
result
final_message = result["messages"][-1].content
print(final_message)


Rikkeisoft được thành lập vào năm 2012. Tính đến hiện tại (năm 2026), Rikkeisoft đã có **14 năm** hoạt động và phát triển.


In [21]:
result

{'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='74cd72b1-b4f8-49c8-b6f6-91b03ac2f0d4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 163, 'total_tokens': 205, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 163}}, 'model_provider': 'openai', 'model_name': 'gemini-3-flash-preview', 'system_fingerprint': None, 'id': 'k1CBadicPOe-vr0P45nj8Qg', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c2122-bc15-79f0-a5fb-9211c2016575-0', tool_calls=[{'name': 'search_rikkeisoft_information', 'args': {'query': 'năm thành lập Rikkeisoft'}, 'id': 'call_6507d2310b1f449cb07210f74929__thought__EjQKMgFyyNp8qPDRrEPFBONkDqA9fu1Z/j6KhmhI1XGn4lGg4cjlMCkXWTGqXepxMOZMTUd5', 'type': 'tool_call'}, {'name': 'get_current_time', 'args': {'utc': 7}, 'id': '

### Dynamic Model là gì?

**Dynamic model** là cơ chế cho phép **chọn mô hình LLM tại thời điểm runtime** dựa trên **ngữ cảnh, trạng thái hiện tại hoặc logic tùy biến** (ví dụ: độ phức tạp câu hỏi, chi phí, độ trễ, người dùng trả phí hay miễn phí).  
Thay vì cố định một model (như GPT-4 hoặc Gemini-Pro), hệ thống có thể **tự động chuyển đổi model** để đạt hiệu quả tốt nhất giữa **chất lượng – chi phí – tốc độ**.

---

### Vì sao cần Dynamic Model?

Dynamic model giúp:
- **Routing thông minh**: câu hỏi đơn giản dùng model rẻ/nhanh, câu hỏi phức tạp dùng model mạnh.
-  **Tối ưu chi phí**: giảm dùng model đắt khi không cần thiết.
-  **Cải thiện hiệu năng**: ưu tiên model phản hồi nhanh trong các tình huống realtime.
- **Linh hoạt mở rộng**: dễ thêm model mới mà không thay đổi toàn bộ hệ thống.

### Middleware với `@wrap_model_call`


Để dùng dynamic model, bạn tạo middleware bằng decorator `@wrap_model_call`. Middleware này sẽ **chỉnh sửa model trong request** trước khi LLM được gọi.


In [26]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatOpenAI(api_key=settings.LLM_API_KEY,model="gemini-5-gemini-2.5-flash", base_url=settings.LLM_BASE_URL)
advanced_model = ChatOpenAI(api_key=settings.LLM_API_KEY,model="gemini-5-gemini-2.5-pro", base_url=settings.LLM_BASE_URL)

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection]
)

In [27]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]}
)
result
final_message = result["messages"][-1].content
print(final_message)

Rikkeisoft được thành lập cách đây 14 năm.


### System Prompt 
**System prompt** là thông điệp dùng để **định hình hành vi, vai trò và phong cách làm việc của agent** ngay từ đầu.  
Nó trả lời cho câu hỏi: *“Agent này nên suy nghĩ và phản hồi như thế nào?”*
Trong LangChain, có thể truyền system prompt khi tạo agent để kiểm soát:
- Cách agent tiếp cận nhiệm vụ
- Mức độ chi tiết / ngắn gọn
- Tính cách, vai trò (assistant, chuyên gia, reviewer, v.v.)

In [28]:
agent = create_agent(
    model=model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection],
    system_prompt="You are a helpful assistant that can answer questions and help with tasks."
)

In [29]:
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time?"}]}
)

{'messages': [HumanMessage(content='what is the current time?', additional_kwargs={}, response_metadata={}, id='d4018b60-53f8-484c-995b-a994ee65bef0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 164, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 32, 'rejected_prediction_tokens': None, 'text_tokens': 12}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 164}}, 'model_provider': 'openai', 'model_name': 'gemini-2.5-flash', 'system_fingerprint': None, 'id': '9FGBafb9PIH_2roP5uai0A8', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c2128-1f2d-71a0-ac82-e3db3619561e-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_2643c3b5d61c4b5fa24d8670c0d0__thought__CrABAXLI2nxNy6VgKYGRYbOmr60mht+YB+xmun/yKOi1vMteO9cNXnhPlmoCVo1oRJT+oVjE4Nzt/tQuYZxjAPZsjy0xq

# Structure Ouput Agent

- ToolStrategy: Sử dụng cho những model không hỗ trợ structure ouput, agent lấy kết quả và tự chuyện sang output mong muốn
- ProviderStrategy: Sử dụng với model hỗ trợ structure output, đáng tin cậy hơn.

In [30]:
from pydantic import BaseModel
from langchain.agents.structured_output import ProviderStrategy
class AgentOutput(BaseModel):
    answer: str
    time_of_answer: str

In [31]:
agent = create_agent(
    model=model,
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    response_format=ProviderStrategy(AgentOutput),
    system_prompt="You are a helpful assistant that can answer questions and help with tasks. You also return the time in the format of YYYY-MM-DD HH:MM:SS that you get from the current time tool."
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]})
result


{'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='90958e2c-419d-4629-b06c-21660084dff3'),
  AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 213, 'total_tokens': 293, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 54, 'rejected_prediction_tokens': None, 'text_tokens': 26}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 213}}, 'model_provider': 'openai', 'model_name': 'gemini-3-flash-preview', 'system_fingerprint': None, 'id': 'AlKBae-aHaCl2roPkPLDuQE', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c2128-5275-73b2-b00c-a9e49fed3ffc-0', tool_calls=[{'name': 'search_rikkeisoft_information', 'args': {'query': 'năm thành lập Rikkeisoft'}, 'id': 'call_372ed81ecb49475993783499d41f__thought__ErUC

In [32]:
print(result["structured_response"])

answer='Rikkeisoft được thành lập vào năm 2012. Tính đến nay (năm 2026), công ty đã hoạt động được 14 năm.' time_of_answer='2026-02-03 08:40:21'


### Memory

Trong LangChain, **Agent tự động duy trì lịch sử hội thoại** thông qua *message state*.  
Phần thông tin này có thể xem như **short-term memory** (bộ nhớ ngắn hạn) của agent, giúp agent:
- Hiểu ngữ cảnh cuộc trò chuyện
- Trả lời nhất quán qua nhiều lượt
- Tham chiếu lại thông tin đã nói trước đó


In [33]:
from langgraph.checkpoint.memory import MemorySaver  
checkpointer = MemorySaver()  # In-memory 
from typing import Any
agent = create_agent(
    model,
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    checkpointer=checkpointer
)
config = {"configurable": {"thread_id": "session_1"}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}],
}, config)

In [34]:
print(result["messages"][-1].content)

Rikkeisoft được thành lập vào ngày **06/04/2012**. Tính đến năm 2026, công ty đã có **14 năm** hình thành và phát triển.

Trong suốt chặng đường này, Rikkeisoft đã vươn mình trở thành một trong những tập đoàn công nghệ hàng đầu Việt Nam, với đội ngũ hàng nghìn nhân sự và có mặt tại nhiều thị trường quốc tế như Nhật Bản, Mỹ, Hàn Quốc và Singapore.


In [35]:
#print checkpointer
checkpointer.get(config)

{'v': 4,
 'ts': '2026-02-03T01:40:54.636693+00:00',
 'id': '1f100a16-13db-6e7e-8007-7a1fc2c7481f',
 'channel_versions': {'__start__': '00000000000000000000000000000002.0.29293118176134336',
  'messages': '00000000000000000000000000000009.0.09803812131322054',
  'branch:to:model': '00000000000000000000000000000009.0.09803812131322054',
  '__pregel_tasks': '00000000000000000000000000000008.0.4133779213276717'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000001.0.39913164274719837'},
  'model': {'branch:to:model': '00000000000000000000000000000008.0.4133779213276717'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='76904cc3-b841-42df-bbe4-b645ed7ce031'),
   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens'

In [36]:
result_2 = agent.invoke({
    "messages": [{"role": "user", "content": "Tôi đã hỏi bạn gì nhỉ?"}],
}, config)
print(result_2["messages"][-1].content)
checkpointer.get(config)


Bạn đã hỏi tôi là: **"Rikkeisoft được thành lập bao nhiêu năm rồi?"**

Tôi đã trả lời rằng Rikkeisoft thành lập vào năm 2012, và tính đến năm 2026 là đã được 14 năm. Bạn có cần thêm thông tin gì về công ty này không?


{'v': 4,
 'ts': '2026-02-03T01:41:02.273447+00:00',
 'id': '1f100a16-5cb0-665d-800a-aa1398e56738',
 'channel_versions': {'__start__': '00000000000000000000000000000011.0.3809122841673662',
  'messages': '00000000000000000000000000000012.0.7476607857359013',
  'branch:to:model': '00000000000000000000000000000012.0.7476607857359013',
  '__pregel_tasks': '00000000000000000000000000000008.0.4133779213276717'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000010.0.00683784578493174'},
  'model': {'branch:to:model': '00000000000000000000000000000011.0.3809122841673662'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='76904cc3-b841-42df-bbe4-b645ed7ce031'),
   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 1

In [38]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Bây giờ là mấy giờ Nhật?"}]
}, config, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

Agent: Bây giờ là mấy giờ Nhật?
Calling tools: ['get_current_time']
Agent: 2026-02-03 10:41:27
Agent: Bây giờ ở Nhật Bản là **10:41 sáng**, Thứ Ba ngày **03/02/2026** (theo múi giờ JST - UTC+9). Nhật Bản nhanh hơn Việt Nam 2 tiếng đồng hồ.
